# LLM Player Research

A working lab notebook for the persistent-session LLM-backed Klondike player. Use it to preserve hypotheses, prompt revisions, run summaries, and qualitative failure analysis before promoting stable results to the root `README.md`.

**Status:** P0 Sol and Luna baselines complete; both 100-game results promoted to the root `README.md`.

## Ten-Game Screening Protocol

Use [`experiments/LLM_EVALUATION.md`](../LLM_EVALUATION.md) as the canonical rubric for new LLM and configuration screens. Run 10 random games, generate the deterministic episode-analysis report, then record the confirmed rating, representative games, dominant failure mode, and next action here. These screens decide whether a model deserves a larger run; they are not promoted as precise win-rate estimates.

## Research Question

Can a current LLM formulate and execute a winning Klondike strategy without receiving the engine's hand-authored strategy?

### Initial hypothesis

The initial self-authored strategy prompt may produce no wins. Meaningful wins may require prompt refinement that helps the model turn its verbal knowledge into consistent play across a complete game.

## Experiment Contract

Keep these conditions fixed unless a run explicitly studies one of them:

- One fresh Codex session per game; never reuse context across deals.
- Ask the model to state the Klondike strategy it already knows before showing the first board.
- Resume the same session for every move so strategy, boards, commands, and feedback accumulate.
- Disable engine guidance and do not send engine-recommended moves.
- Generate legal moves in the Java engine and constrain the response schema to that set.
- Authenticate Codex CLI through the ChatGPT subscription, not an API key.
- Record exact model ID, reasoning effort, Codex CLI version, prompt version, random-deal method, and maximum move limit.
- Use a fresh independent random shuffle for every game; obtain confidence through sample size rather than replaying a fixed deal suite.

In [ ]:
from pathlib import Path
import json

# Adjust only when the experiment contract intentionally changes.
experiment_contract = {
    "session_mode": "persistent_per_game",
    "strategy_source": "model_authored_before_first_board",
    "engine_guidance": False,
    "access_path": "codex_cli_chatgpt_subscription",
    "max_moves": 1000,
    "confidence_level": 0.95,
}
experiment_contract

## Prompt Version Ledger

Assign a version before every prompt change. Preserve the exact text or link to the commit that contains it. Record the expected effect before running games.

| Version | Description | Expected effect | Source |
|---|---|---|---|
| P0 | Ask what Klondike rules and strategy the model already knows; provide only board notation and legal moves during play | Establish an unassisted persistent-session baseline | `CodexCliPlayer.STRATEGY_PROMPT` |
| P1 | TBD | TBD | TBD |

## Run Ledger

One row represents one fixed combination of model, reasoning effort, prompt version, and deal suite. Do not combine pilot runs with confirmatory runs unless their contracts are identical.

| Run ID | Date | Model | Reasoning | Prompt | Deals | Games | Wins | Win % and 95% CI | Avg time | Avg moves | Notes |
|---|---|---|---|---|---|---:|---:|---|---|---|---|
| `p0-sol-medium-smoke-20260924` | 2026-09-24 | `gpt-5.6-sol` | Medium | P0 | Independent random shuffle | 1 | 0 | 0.00% (n=1; interval pending) | 480.706s | 60.00 | Protocol succeeded; model quit with only potentially repetitive tableau moves remaining |
| `p0-sol-medium-100-20260925` | 2026-09-24 to 2026-09-25 | `gpt-5.6-sol` | Medium | P0 | Independent random shuffle | 100 | 18 | 18.00% ± 7.53% | 474.557s | 59.68 | Completed across 10, 1, 9, and 80-game batches; two interrupted partial games excluded |
| `p0-luna-medium-100-20260926` | 2026-09-25 to 2026-09-26 | `gpt-5.6-luna` | Medium | P0 | Independent random shuffle | 100 | 5 | 5.00% ± 4.27% | 890.632s | 130.78 | First 97 completed games plus a 3-game completion batch; one interrupted partial game excluded |

In [ ]:
# Future result runners should emit one JSON object per configuration.
# Keep generated runtime data out of git; promote only small, reviewed records.
result_candidates = [
    Path("experiments/runtime/artifacts/llm-player-results.jsonl"),
    Path("../runtime/artifacts/llm-player-results.jsonl"),
]
results_path = next((path for path in result_candidates if path.exists()), None)
records = []
if results_path:
    records = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]
records

## Failure Taxonomy

Classify failed games so prompt changes respond to observed behavior rather than intuition alone.

- **Premature foundation move:** loses a card needed for tableau development.
- **Missed reveal:** chooses progress that does not uncover a face-down tableau card.
- **Poor empty-column use:** creates or fills an empty column without a productive King plan.
- **Stock-memory failure:** forgets previously seen waste cards or the order of a stock pass.
- **Cycle:** returns to repeated board states without new information or progress.
- **Short-horizon choice:** selects an attractive immediate move that blocks a known follow-up.
- **Premature quit:** stops while productive legal play remains.
- **Context drift:** stops following its stated strategy later in a long game.
- **Protocol failure:** returns malformed output or a command outside the legal set.
- **Move-limit termination:** reaches the configured safety cap.

## Run Notes Template

Copy this block for each meaningful run:

### Run `<run-id>`: `<short description>`

- **Question:**
- **Prediction made before running:**
- **Configuration:**
- **Result:**
- **Representative Codex session IDs:**
- **Observed failure modes:**
- **Interpretation:**
- **Decision for the next prompt version:**

### Run `p0-sol-medium-smoke-20260924`: first persistent-session smoke game

- **Question:** Can the unassisted P0 model complete a game, and will persistent session resumption plus the response schema work for a full game?
- **Prediction made before running:** The model may not win; command formatting may fail before strategy can be evaluated.
- **Configuration:** `gpt-5.6-sol`, medium reasoning, Codex CLI 0.154.0, P0, persistent session, engine guidance disabled, independent random shuffle, 1,000-move cap.
- **Result:** 0/1 wins, 60 moves, 480.706 seconds. The Gradle result test passed.
- **Codex session ID:** `01a0d001-aeee-7cd2-b030-23c4ba86cabe`.
- **Protocol observation:** All 60 move responses satisfied the per-turn JSON schema and selected a listed legal command. Session creation and every resume succeeded.
- **Strategy observation:** The model produced a strong verbal strategy covering hidden-card reveals, safe foundation play, empty columns, stock memory, and short combinations.
- **Observed behavior:** Many stock turns. On the final board the model chose `quit` while three non-quit tableau moves remained: `move T3 8♥ T4`, `move T3 5♣ T5`, and `move T6 6♠ T7`. At least the last option was the exact inverse of an earlier `move T7 6♠ T6`; the legal-move generator does not filter repeats or visited states.
- **Interpretation:** Output formatting is not the first blocker. The remaining moves may all have been reversible rearrangements, so this run does not establish that quitting was strategically wrong. One random game is diagnostic only and says nothing reliable about win rate.
- **Decision for the next prompt version:** Do not change P0 based on this result alone. Continue with independent random games and use session transcripts plus terminal-state diagnostics to understand repeated failure patterns.

### Run `p0-sol-medium-100-20260925`: unassisted persistent-session baseline

- **Question:** Can `gpt-5.6-sol` formulate and execute a winning Klondike strategy without the engine's hand-authored strategy?
- **Prediction made before running:** P0 might produce no wins and require significant prompt refinement before the model could convert its verbal strategy into successful play.
- **Configuration:** `gpt-5.6-sol`, medium reasoning, Codex CLI 0.154.0, P0, one persistent session per game, engine guidance disabled, independent random shuffles, 1,000-move cap, and episode logging enabled.
- **Result:** 18/100 wins (18.00% ± 7.53% using the repository's normal-approximation notation; Wilson 95% interval approximately 11.7%-26.7%), 59.68 average moves, 474.557 average seconds per game, 47,455.715 total seconds, and a best win streak of 2.
- **Batch composition:** 10 completed games, then 1 and 9 completed games from two interrupted extension attempts, followed by a successful 80-game batch. The two games interrupted by infrastructure failures have no `EPISODE_SUMMARY` and are excluded.
- **Protocol observation:** All recorded commands belonged to their engine-generated legal-move sets. The final 80-game batch encountered eight transient failed CLI attempts; bounded retries recovered each one and the batch completed successfully.
- **Artifacts:** `engine/logs/game.log` contains model, reasoning, session ID, outcome, move count, and duration per completed game. `engine/logs/episode.log` contains exactly 100 completed summaries plus move-level state records; Codex session transcripts remain in the local Codex session store.
- **Interpretation:** The initial hypothesis was not supported. Sol won without receiving the engine's hand-authored strategy, showing that self-authored strategy plus persistent turn context is sufficient for meaningful play. The 18% estimate remains substantially below A* search and is dramatically slower, but it exceeds the historical 13% stateless API result in this sample. The experiments are not direct prompt or model comparisons.
- **Decision for the next prompt version:** Preserve P0 as the baseline. Analyse failures and terminal positions before proposing P1 or varying reasoning effort; do not retrospectively add strategy to this result.

### Run `p0-luna-medium-100-20260926`: unassisted persistent-session baseline

- **Question:** Can the lower-cost `gpt-5.6-luna` model formulate and execute a winning Klondike strategy under the same P0 persistent-session approach?
- **Configuration:** `gpt-5.6-luna`, medium reasoning, P0, one persistent session per game, engine guidance disabled, independent random shuffles, a 200-move cap, and episode logging enabled.
- **Result:** 5/100 wins (5.00% ± 4.27% using the repository's normal-approximation notation; Wilson 95% interval approximately 2.2%-11.2%), 130.78 average moves, 890.632 average seconds per game, 89,063.176 total seconds, and a best win streak of 1.
- **Winning games:** Games 5, 17, 46, 57, and 96 in the combined completed-game sequence.
- **Batch composition:** The original run completed games 1-97. Its game 98 stopped after 56 recorded steps during repeated Codex CLI failures and has no `EPISODE_SUMMARY`, so it is excluded. A separate three-game batch completed the sample; all three were losses.
- **Artifacts:** The original summaries and the three completion summaries are in `engine/logs/episode.log`; the completion batch has run ID `auto-37f359ad-68e5-4719-8f2c-e648b3d0cdcf`. Per-game results and Codex session IDs are in `engine/logs/game.log`.
- **Interpretation:** Luna can win with a self-authored strategy and no engine guidance, but its 5% result is materially below Sol's 18% result in these samples. The runs are not a fully controlled model comparison because Luna used a 200-move cap while Sol used a 1,000-move cap.
- **Decision for the next prompt version:** Preserve this result as the Luna P0 baseline. Apply the ten-game screening and episode-log rubric before committing Luna or another model to a larger follow-up run.

## Prompt Refinement Guardrails

- Write down the predicted failure and expected improvement before editing the prompt.
- Change one conceptual element at a time when practical.
- Preserve old prompt versions and their results.
- Do not tune against one unusually easy or difficult random deal.
- Separate small exploratory pilots from larger random-sample evaluations.
- Treat model, reasoning effort, prompt, memory policy, and engine guidance as separate variables.
- Record unsuccessful prompt versions; they are part of the research result.

## Promotion Checklist

Move a result into the root `README.md` only when:

- The prompt version and experiment contract are frozen.
- The model was evaluated with the documented random-deal method and sample size.
- The sample size and confidence interval are reported.
- Aggregate results can be regenerated from machine-readable records.
- Session IDs or equivalent transcripts exist for qualitative review.
- Pilot and confirmatory runs are clearly distinguished.
- The README row names the exact model, reasoning effort, prompt version, and persistent-session behavior.

## Conclusions

The P0 baselines demonstrate that current Codex CLI models can turn a self-authored Klondike strategy into successful play without engine guidance. Sol won 18 of 100 independent random games and Luna won 5 of 100 while maintaining a persistent session for each deal. This rejects the initial expectation that wins would require substantial prompt refinement, while showing substantial variation between models. The result does not isolate persistent context, model generation, or prompt style, and the Sol and Luna runs used different move caps. A* remains stronger at 36.87% and much faster. P0 should remain frozen while the episode traces are used to classify failures and choose the next single experimental variable.